# Tutorial completo da `dispropy` 0.5.0

Este notebook apresenta as funcionalidades públicas da biblioteca para análise
de desproporcionalidade em farmacovigilância:

- Reporting Odds Ratio (ROR);
- Proportional Reporting Ratio (PRR);
- Information Component (IC, IC025 e IC975);
- Empirical Bayes Geometric Mean (EBGM, EB05 e EB95);
- flags de triagem, seleção de métricas e diagnósticos do ajuste GPS;
- validação dos dados e comportamento de `inplace`.

> **Atenção:** desproporcionalidade estatística não demonstra causalidade, não
> estima incidência e não substitui avaliação clínica e farmacológica.

## Instalação

Na raiz do repositório, instale a biblioteca em modo editável:

```bash
python -m pip install -e .
```

O notebook usa apenas dependências da própria biblioteca: pandas, NumPy e SciPy.

In [1]:
import warnings

import numpy as np
import pandas as pd

import dispropy as disp
from dispropy import GPSFitWarning, calculate_disproportionality

print("dispropy", disp.__version__)

dispropy 0.5.0


## 1. Tabela 2x2

Cada linha representa um par medicamento-evento:

|                         | Evento de interesse | Outros eventos |
|-------------------------|---------------------|----------------|
| Medicamento de interesse | A                  | B              |
| Outros medicamentos      | C                  | D              |

Os nomes das colunas são livres. A biblioteca valida tipo numérico, ausência
de valores faltantes, não negatividade e total maior que zero. A validação não
confirma que a tabela foi construída epidemiologicamente de forma correta.

In [2]:
df = pd.DataFrame({
    "medicamento": ["Drug A", "Drug A", "Drug B", "Drug B"],
    "evento": ["Event W", "Event X", "Event Y", "Event Z"],
    "A": [10, 5, 20, 2],
    "B": [90, 95, 80, 98],
    "C": [20, 25, 30, 20],
    "D": [880, 875, 870, 880],
})
df

,medicamento,evento,A,B,C,D
0,Drug A,Event W,10,90,20,880
1,Drug A,Event X,5,95,25,875
2,Drug B,Event Y,20,80,30,870
3,Drug B,Event Z,2,98,20,880


## 2. ROR, PRR e IC em uma única chamada

In [3]:
resultado = calculate_disproportionality(
    df,
    a_col="A",
    b_col="B",
    c_col="C",
    d_col="D",
    correction=0.5,
    shrinkage=0.5,
    add_signal_flags=True,
)

colunas = [
    "medicamento", "evento", "ror", "ror_lower_95", "ror_upper_95",
    "prr", "prr_lower_95", "prr_upper_95", "expected_count",
    "ic", "ic025", "ic975", "signal_ror", "signal_prr", "signal_ic",
]
resultado[colunas].round(4)

,medicamento,evento,ror,ror_lower_95,ror_upper_95,prr,prr_lower_95,prr_upper_95,expected_count,ic,ic025,ic975,signal_ror,signal_prr,signal_ic
0,Drug A,Event W,4.9833,2.2966,10.8128,4.5692,2.2356,9.3386,3.0,1.5850,0.5078,2.3109,True,True,True
1,Drug A,Event X,1.9773,0.7683,5.0892,1.9241,0.7837,4.7242,3.0,0.6521,-0.9101,1.6367,False,False,False
2,Drug B,Event Y,7.2682,3.9708,13.3040,5.9959,3.5625,10.0916,5.0,1.8981,1.1477,2.4228,True,True,True
3,Drug B,Event Z,1.0901,0.2885,4.1195,1.0879,0.2974,3.9791,2.2,-0.1110,-2.7041,1.2804,False,False,False


As flags são critérios de triagem:

- `signal_ror`: limite inferior de 95% do ROR maior que 1;
- `signal_prr`: limite inferior de 95% do PRR maior que 1;
- `signal_ic`: IC025 maior que 0.

Elas não devem ser interpretadas como confirmação de reação adversa causal.

## 3. Funções individuais e API curta

Por padrão, `correction=0.0`: `disp.ror` e `disp.prr` calculam sobre as
contagens originais da tabela, sem correção de continuidade. Para aplicar a
correção convencional (soma 0,5 às quatro células A, B, C e D), passe
`correction=0.5` explicitamente — como foi feito na chamada de
`calculate_disproportionality` na seção anterior.


In [4]:
ror = disp.ror(df, "A", "B", "C", "D")
prr = disp.prr(df, "A", "B", "C", "D")
ic = disp.ic(df, "A", "B", "C", "D")

pd.DataFrame({
    "ror": ror["ror"],
    "prr": prr["prr"],
    "ic": ic["ic"],
    "ic025": ic["ic025"],
}).round(4)

,ror,prr,ic,ic025
0,4.8889,4.5,1.5850,0.5078
1,1.8421,1.8,0.6521,-0.9101
2,7.2500,6.0,1.8981,1.1477
3,0.8980,0.9,-0.1110,-2.7041


## 4. Calcular somente métricas selecionadas

In [5]:
somente_ror = disp.disproportionality(
    df, "A", "B", "C", "D", metrics=("ror",)
)

[coluna for coluna in somente_ror.columns if coluna.startswith("ror") or coluna.startswith("prr") or coluna == "ic"]

['ror', 'ror_lower_95', 'ror_upper_95']

## 5. Nomes personalizados para A, B, C e D

In [6]:
df_nomes = df.rename(columns={
    "A": "drug_event",
    "B": "drug_other",
    "C": "other_drugs_event",
    "D": "other_drugs_other",
})

disp.ror(
    df_nomes,
    a_col="drug_event",
    b_col="drug_other",
    c_col="other_drugs_event",
    d_col="other_drugs_other",
)[["medicamento", "evento", "ror"]].round(4)

,medicamento,evento,ror
0,Drug A,Event W,4.8889
1,Drug A,Event X,1.8421
2,Drug B,Event Y,7.2500
3,Drug B,Event Z,0.8980


## 6. Cópia versus alteração no próprio DataFrame

Por padrão, `inplace=False`: o original não é modificado. Com `inplace=True`,
a função modifica e retorna o mesmo objeto.

In [7]:
original = df.copy()
copia_calculada = disp.ror(original, "A", "B", "C", "D")
print("ROR no original após inplace=False:", "ror" in original.columns)

retornado = disp.ror(original, "A", "B", "C", "D", inplace=True)
print("ROR no original após inplace=True:", "ror" in original.columns)
print("Mesmo objeto:", retornado is original)

ROR no original após inplace=False: False
ROR no original após inplace=True: True
Mesmo objeto: True


## 7. EBGM com conjunto adequado para ajuste global

O GPS estima cinco hiperparâmetros usando todos os pares simultaneamente. O
exemplo abaixo gera 300 pares sintéticos heterogêneos e reproduzíveis. Em dados
reais, cada linha deve representar um par válido da mesma base analisada.

In [8]:
rng = np.random.default_rng(20260702)
size = 300
expected_target = rng.uniform(0.5, 25.0, size=size)
component = rng.random(size) < 0.65
relative_rate = np.empty(size)
relative_rate[component] = rng.gamma(1.2, 1 / 1.5, component.sum())
relative_rate[~component] = rng.gamma(4.0, 1 / 1.2, (~component).sum())
observed = rng.poisson(expected_target * relative_rate)

total = np.full(size, 100_000, dtype=int)
drug_total = rng.integers(500, 5_000, size=size)
event_total = np.maximum(
    observed + 1,
    np.rint(expected_target * total / drug_total).astype(int),
)

df_gps = pd.DataFrame({
    "par": [f"Pair {i + 1}" for i in range(size)],
    "A": observed.astype(int),
    "B": drug_total - observed,
    "C": event_total - observed,
    "D": total - drug_total - event_total + observed,
})
df_gps.head()

,par,A,B,C,D
0,Pair 1,24,3810,226,95940
1,Pair 2,3,2949,369,96679
2,Pair 3,72,4812,361,94755
3,Pair 4,0,4003,134,95863
4,Pair 5,83,2541,651,96725


In [9]:
resultado_ebgm = disp.ebgm(df_gps, "A", "B", "C", "D")

resultado_ebgm[
    ["par", "observed_count", "expected_count", "qn", "ebgm", "eb05", "eb95"]
].sort_values("ebgm", ascending=False).head(10).round(4)

,par,observed_count,expected_count,qn,ebgm,eb05,eb95
194,Pair 195,45.0,5.4510,0.9999,7.1627,5.6351,9.0004
176,Pair 177,115.0,15.5240,0.9999,7.0359,6.0381,8.1597
85,Pair 86,40.0,5.1464,0.9998,6.7342,5.2268,8.5658
277,Pair 278,145.0,21.6209,0.9999,6.4764,5.6500,7.3953
138,Pair 139,67.0,9.5900,0.9998,6.4688,5.3030,7.8282
143,Pair 144,127.0,19.5889,0.9998,6.2461,5.3993,7.1945
155,Pair 156,123.0,19.6482,0.9998,6.0387,5.2080,6.9707
299,Pair 300,63.0,10.2588,0.9995,5.7563,4.6908,7.0043
14,Pair 15,16.0,2.1820,0.9984,5.6135,3.8405,7.9750
172,Pair 173,99.0,17.5780,0.9994,5.4336,4.6091,6.3705


### Parâmetros e diagnósticos do modelo GPS

In [10]:
resultado_ebgm.attrs["gps_model"]

{'alpha1': 4.790695330938531,
 'beta1': 1.43058229228602,
 'alpha2': 1.44419596696585,
 'beta2': 2.372864541160953,
 'weight': 0.36131481772870866,
 'converged': True,
 'parameters_near_bounds': False,
 'near_bound_parameters': [],
 'valid_pair_count': 300,
 'recommended_min_valid_pairs': 50,
 'log_likelihood': -1119.98956267684,
 'n_star': 0}

Os campos mais importantes para confiabilidade são:

- `converged`: estado informado pelo otimizador;
- `parameters_near_bounds`: indica proximidade de algum limite de otimização;
- `near_bound_parameters`: nomes dos parâmetros afetados;
- `valid_pair_count`: pares usados para ajustar o modelo;
- `recommended_min_valid_pairs`: limiar operacional de aviso.

### EBGM dentro da função principal e flag de sinal

In [11]:
resultado_completo = disp.disproportionality(
    df_gps,
    "A", "B", "C", "D",
    metrics=("ror", "prr", "ic", "ebgm"),
    add_signal_flags=True,
)

resultado_completo[
    ["par", "ror", "prr", "ic", "ebgm", "eb05", "eb95", "signal_ebgm"]
].sort_values("ebgm", ascending=False).head(10).round(4)

,par,ror,prr,ic,ebgm,eb05,eb95,signal_ebgm
194,Pair 195,11.8901,11.7660,2.9347,7.1627,5.6351,9.0004,True
176,Pair 177,10.1149,9.8320,2.8496,7.0359,6.0381,8.1597,True
85,Pair 86,10.0558,9.9397,2.8425,6.7342,5.2268,8.5658,True
277,Pair 278,8.2373,7.7995,2.7175,6.4764,5.6500,7.3953,True
138,Pair 139,8.5893,8.4037,2.7420,6.4688,5.3030,7.8282,True
143,Pair 144,8.9011,8.6731,2.6660,6.2461,5.3993,7.1945,True
155,Pair 156,7.6507,7.3530,2.6158,6.0387,5.2080,6.9707,True
299,Pair 300,8.5009,8.4060,2.5612,5.7563,4.6908,7.0043,True
14,Pair 15,8.5946,8.5390,2.6211,5.6135,3.8405,7.9750,True
172,Pair 173,6.5542,6.3124,2.4605,5.4336,4.6091,6.3705,True


`signal_ebgm` é verdadeiro quando `EB05 > 2` e `A >= 3`. É uma flag de
triagem, não uma conclusão causal.

## 8. Entendendo `GPSFitWarning`

O exemplo pequeno abaixo é deliberadamente inadequado para ajustar cinco
hiperparâmetros. A biblioteca ainda calcula o resultado, mas avisa sobre poucos
pares válidos e parâmetros próximos aos bounds.

In [12]:
df_gps_pequeno = pd.DataFrame({
    "A": [0, 1, 2, 5, 10, 20, 40, 3],
    "B": [100, 99, 98, 95, 90, 80, 60, 97],
    "C": [5, 10, 20, 25, 20, 30, 40, 50],
    "D": [895, 890, 880, 875, 880, 870, 860, 850],
})

with warnings.catch_warnings(record=True) as alertas:
    warnings.simplefilter("always", GPSFitWarning)
    resultado_instavel = disp.ebgm(df_gps_pequeno, "A", "B", "C", "D")

for alerta in alertas:
    print(f"{alerta.category.__name__}: {alerta.message}")
    print()

resultado_instavel.attrs["gps_model"]

{'alpha1': 100.0,
 'beta1': 23.599551977006055,
 'alpha2': 90.60803195279887,
 'beta2': 100.0,
 'weight': 0.4111649284291886,
 'converged': True,
 'parameters_near_bounds': True,
 'near_bound_parameters': ['alpha1', 'beta2'],
 'valid_pair_count': 8,
 'recommended_min_valid_pairs': 50,
 'log_likelihood': -20.422421658283348,
 'n_star': 0}

Quando esses avisos aparecerem:

1. confira a construção de A/B/C/D e a diversidade dos pares;
2. use um conjunto maior e representativo para estimar a prior GPS;
3. examine `near_bound_parameters`;
4. realize análise de sensibilidade ou validação estatística independente;
5. não use o EBGM isoladamente para decisão clínica ou regulatória.

## 9. Exemplos de validação e mensagens de erro

In [13]:
casos_invalidos = {
    "coluna ausente": df.drop(columns="D"),
    "valor negativo": df.assign(A=[-1, 5, 20, 2]),
    "valor ausente": df.assign(A=[np.nan, 5, 20, 2]),
    "coluna não numérica": df.assign(A=["10", "5", "20", "2"]),
}

for descricao, dados in casos_invalidos.items():
    try:
        disp.ror(dados, "A", "B", "C", "D")
    except (TypeError, ValueError) as erro:
        print(f"{descricao}: {type(erro).__name__}: {erro}")

coluna ausente: ValueError: Contingency column(s) not found: ['D'].
valor negativo: ValueError: Contingency columns must contain values greater than or equal to zero.
valor ausente: ValueError: Contingency columns must not contain missing values.
coluna não numérica: TypeError: Contingency column(s) must be numeric: ['A'].


## 10. Checklist antes de interpretar

- A unidade de contagem foi definida e aplicada consistentemente?
- Duplicidades e seguimentos da mesma notificação foram tratados?
- As definições de medicamento, evento e comparador são adequadas?
- Todas as linhas pertencem à mesma base e período analítico?
- Contagens pequenas e intervalos amplos foram considerados?
- Os warnings e metadados do GPS foram revisados?
- O achado passou por avaliação clínica, farmacológica e de vieses?

Este notebook demonstra a API. Um protocolo real deve definir previamente a
construção das tabelas, população comparadora, critérios de sinal e análises de
sensibilidade.